# Mol* visualization catalog

Progress dashboard for the molstarLib migration (legacy `deeporigin-molstar` → hosted
`molstarLib` at `os.dev.deeporigin.io`).

| # | Visualization | Status |
|---|---------------|--------|
| 1 | Protein structure | ✅ migrated |
| 2 | Protein + binding pockets | ✅ migrated |
| 3 | Single ligand 3D | ✅ migrated |
| 4 | Ligand set 3D | ⏳ legacy (multi-mol SDF not yet in molstarLib) |
| 5 | Protein + docked poses | ✅ migrated |
| 6 | Protein + pockets + poses | ✅ migrated |
| 7 | Docking search box | ✅ migrated |
| 8 | Protein + box + poses | ✅ migrated |
| 9 | MD trajectory | ⏳ legacy |

**Requires:** sections 1–3 and 5–8 use the hosted molstarLib bundle only (no `tools` extra).
Sections 4 and 9 still need `uv sync --extra tools` (legacy `deeporigin-molstar`).
Section 9 also needs a completed ABFE execution.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from deeporigin.drug_discovery import BRD_DATA_DIR, Ligand, LigandSet, Pocket, Protein
from deeporigin.viz.molstar_html import MOLSTAR_JS_URL


def _find_repo_root() -> Path:
    """Return the CLI repo root whether the notebook cwd is repo root or docs/notebooks/."""
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "tests").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find CLI repo root. Run this notebook from the cli repo "
        "(repo root or docs/notebooks/dirty/)."
    )


REPO_ROOT = _find_repo_root()
POCKET_FIXTURE = REPO_ROOT / "tests/fixtures/files/pocketfinder/pocket_1.pdb"

print(f"Repo root: {REPO_ROOT}")
print(f"Mol* bundle URL: {MOLSTAR_JS_URL}")
print(f"Pocket fixture: {POCKET_FIXTURE} ({'ok' if POCKET_FIXTURE.is_file() else 'missing'})")

## 1. Protein structure — ✅ migrated

Entry point: `Protein.show()` (no pockets or poses).

**Check:** cartoon representation, Mol* UI loads, no iframe console errors.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.show()

## 2. Protein + binding pockets — ✅ migrated

Entry point: `Protein.show(pockets=...)`.

**Check:** semi-transparent pocket surfaces (alpha ~0.25), protein cartoon with faint
surface (alpha ~0.1), pocket colors match `Pocket.color`.

In [ ]:
pocket = Pocket.from_pdb_file(POCKET_FIXTURE, name="pocket-1", color="red")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 30.0
pocket.get_center()

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.show(pockets=[pocket])

## 3. Single ligand 3D — ✅ migrated

Entry point: `Ligand.show()`.

**Check:** ball-and-stick ligand in Mol* viewer via hosted `molstarLib`
(`loadFromRawContent` on SDF).

In [ ]:
ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand.show()

## 4. Ligand set 3D — ⏳ legacy

Entry point: `LigandSet.show()`.

**Check:** multiple ligands from a combined SDF via legacy
`deeporigin_molstar.MoleculeViewer` (molstarLib does not yet split multi-mol SDF).

Requires `uv sync --extra tools`.

In [ ]:
ligands = LigandSet.from_dir(BRD_DATA_DIR)
ligands.show()

## 5. Protein + docked poses — ✅ migrated

Entry point: `Protein.show(poses=...)`.

**Check:** protein cartoon with all docked poses overlaid (`visualizeDockedLigands`).
A navigation bar appears at the bottom when there is more than one pose: use the
◀ / ▶ buttons or the Left/Right arrow keys to cycle through "all poses" and each
individual pose (`showLigandAtIndex`). Uses BRD ligands as stand-in poses.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
poses = LigandSet.from_dir(BRD_DATA_DIR)
protein.show(poses=poses)

## 6. Protein + pockets + poses — ✅ migrated

Entry point: `Protein.show(pockets=..., poses=...)`.

**Check:** pocket surfaces and docked poses together
(`renderStructureWithPocketsAndLigands`). The same bottom navigation bar (◀ / ▶
buttons or Left/Right arrow keys) steps through all poses and each individual pose.

In [ ]:
pocket = Pocket.from_pdb_file(POCKET_FIXTURE, name="pocket-1", color="red")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 30.0
pocket.get_center()

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
poses = LigandSet.from_dir(BRD_DATA_DIR)
protein.show(pockets=[pocket], poses=poses)

## 7. Docking search box — ✅ migrated

Entry point: `Docking.show_box()`.

**Check:** protein with yellow wireframe bounding box from pocket center and box
dimensions (`loadFromRawContent` + `renderBoundingBox`).

Note: box geometry uses a duck-typed `{min,max}` until
[PUI-2203](https://deeporigin.atlassian.net/browse/PUI-2203) exports `createBox3D`.

In [ ]:
from deeporigin.drug_discovery import Docking

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
# Docking() requires protein.id; use a placeholder for local visualization only.
if protein.id is None:
    protein.id = "notebook-demo-protein"

ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
pocket = Pocket.from_pdb_file(POCKET_FIXTURE, name="pocket-1")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 15.0
pocket.get_center()

docking = Docking(protein=protein, pocket=pocket, ligand=ligand)
docking.show_box()

## 8. Protein + box + poses — ✅ migrated

Entry point: `Docking.show_box(poses=...)`.

**Check:** protein cartoon, yellow wireframe docking box, and ligand pose(s)
overlaid (`visualizeDockedLigands` + `renderBoundingBox`). Uses the input
ligand as a stand-in pose for local visualization (not a real docking output).

Note: same duck-typed `{min,max}` box geometry as section 7 until
[PUI-2203](https://deeporigin.atlassian.net/browse/PUI-2203).

In [ ]:
from deeporigin.drug_discovery import Docking

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
# Docking() requires protein.id; use a placeholder for local visualization only.
if protein.id is None:
    protein.id = "notebook-demo-protein"

ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
pocket = Pocket.from_pdb_file(POCKET_FIXTURE, name="pocket-1")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 15.0
pocket.get_center()

docking = Docking(protein=protein, pocket=pocket, ligand=ligand)
# Overlay the input ligand as a stand-in pose with the search box.
docking.show_box(poses=ligand)

## 9. MD trajectory — ⏳ legacy

Entry point: `ABFE.show_trajectory()`.

**Check:** trajectory playback in Mol* with protein + XTC frames.

Requires a **completed ABFE execution** with trajectory paths in results. Run the
[ABFE notebook](./abfe.ipynb) first, then uncomment and set `abfe` below.

In [ ]:
# from deeporigin.drug_discovery import ABFE
#
# abfe = ABFE.from_id("<completed-abfe-execution-id>")
# abfe.show_trajectory(step="binding", window=1)